In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

In [2]:
# load data
df_att = pd.read_csv('attendance data with features.csv')

rand_seed = 34

In [3]:
# remove unnecessary features
# duplicates, linearly dependent variables, etc.

remove = ['HGP', 'VGP',
          'HPTS', 'VPTS',
          'HGF', 'VGF',
          'HGA', 'VGA',
          'HPP%', 'VPP%', 
          'HPK%', 'HPK%',
          'HS', 'VS', 
          'HSA', 'VSA']

df_att = df_att.drop(columns = remove)

In [4]:
# A, PA, CAP
df_cap = df_att.copy().drop(columns = ['LA', 'LCAP'])

# LA, LCAP
df_lcap = df_att.copy().drop(columns = ['A', 'PA','CAP'])

X_cap_a = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_a = df_att['A']

X_cap_pa = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'LCAP'])
y_cap_pa = df_att['PA']

X_lcap = df_att.drop(columns = ['H', 'V', 'A', 'LA', 'PA', 'CAP'])
y_lcap = df_att['LA']

In [5]:
# Function to compute evaluation metrics with dependent variable transformation
def compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred):
    if dep == 'A':
        y_train_true, y_train_pred = y_train, y_train_pred
        y_test_true, y_test_pred = y_test, y_test_pred

    elif dep == 'LA':
        y_train_true, y_train_pred = np.exp(y_train), np.exp(y_train_pred)
        y_test_true, y_test_pred = np.exp(y_test), np.exp(y_test_pred)

    elif dep == 'PA':
        y_train_true, y_train_pred = y_train * X_train['CAP'], y_train_pred * X_train['CAP']
        y_test_true, y_test_pred = y_test * X_test['CAP'], y_test_pred * X_test['CAP']

    return {
        "RMSE Train": sqrt(mean_squared_error(y_train_true, y_train_pred)),
        "MAE Train": mean_absolute_error(y_train_true, y_train_pred),
        "R² Train": r2_score(y_train_true, y_train_pred),
        "RMSE Test": sqrt(mean_squared_error(y_test_true, y_test_pred)),
        "MAE Test": mean_absolute_error(y_test_true, y_test_pred),
        "R² Test": r2_score(y_test_true, y_test_pred)
    }

In [6]:
# Compute Pearson and Spearman correlation matrices
corr_matrix_pearson = df_att.drop(columns=['H', 'V']).corr(method='pearson')
corr_matrix_spearman = df_att.drop(columns=['H', 'V']).corr(method='spearman')

# Function to plot and save heatmap
def plot_heatmap(corr_matrix, title, save_path):
    plt.figure(figsize=(15, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, square=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=7, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=7, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')  # Save the figure
    plt.close()  # Close the plot to free memory

# Save Pearson heatmap
plot_heatmap(corr_matrix_pearson, 'Pearson Correlation Heatmap', 'pearson_heatmap.png')

# Save Spearman heatmap
plot_heatmap(corr_matrix_spearman, 'Spearman Correlation Heatmap', 'spearman_heatmap.png')

In [7]:
# Compute the difference between Pearson and Spearman correlations
corr_diff = corr_matrix_pearson - corr_matrix_spearman

# Plot and save the heatmap of the differences
plt.figure(figsize=(15, 10))
sns.heatmap(corr_diff, cmap='coolwarm', center=0, square=True)
plt.title('Difference Between Pearson and Spearman Correlations')
plt.xticks(ticks=range(len(corr_diff.columns)), labels=corr_diff.columns, fontsize=7, rotation=90)
plt.yticks(ticks=range(len(corr_diff.index)), labels=corr_diff.index, fontsize=7, rotation=0)
plt.savefig('correlation_difference_heatmap.png', dpi=300, bbox_inches='tight')  # Save the figure
plt.close()

In [8]:
# Define dependent variable(s)
dependent_vars = ['A', 'LA', 'PA']  # Replace with actual dependent variable(s)

# Get only independent variables
independent_vars = [col for col in df_att.columns if col not in dependent_vars + ['H', 'V']]

# Compute Pearson and Spearman correlations for only the dependent variable(s)
corr_pearson_dep = df_att.drop(columns=['H', 'V']).corr(method='pearson').loc[dependent_vars, independent_vars]
corr_spearman_dep = df_att.drop(columns=['H', 'V']).corr(method='spearman').loc[dependent_vars, independent_vars]

# Function to plot heatmap for dependent variables
def plot_heatmap_dep(corr_matrix, title, save_path):
    plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
    sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, cbar=True)
    plt.title(title)
    plt.xticks(ticks=range(len(corr_matrix.columns)), labels=corr_matrix.columns, fontsize=8, rotation=90)
    plt.yticks(ticks=range(len(corr_matrix.index)), labels=corr_matrix.index, fontsize=8, rotation=0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# Save heatmaps for dependent variables
plot_heatmap_dep(corr_pearson_dep, 'Pearson Correlation (Dependent Variables)', 'pearson_dep_heatmap.png')
plot_heatmap_dep(corr_spearman_dep, 'Spearman Correlation (Dependent Variables)', 'spearman_dep_heatmap.png')

In [9]:
##### Compute the difference for only the dependent variable(s)
corr_diff_dep = corr_pearson_dep - corr_spearman_dep

# Plot and save heatmap for the difference
plt.figure(figsize=(10, len(dependent_vars)))  # Adjust height dynamically
sns.heatmap(corr_diff_dep, cmap='coolwarm', center=0, cbar=True)
plt.title('Difference Between Pearson and Spearman Correlations (Dependent Variables)')
plt.xticks(ticks=range(len(corr_diff_dep.columns)), labels=corr_diff_dep.columns, fontsize=8, rotation=90)
plt.yticks(ticks=range(len(corr_diff_dep.index)), labels=corr_diff_dep.index, fontsize=8, rotation=0)
plt.savefig('correlation_difference_dep_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

In [10]:
# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# variance inflation factor
# Select numerical columns (excluding target variable)
df_numeric = df_att.drop(columns=['H', 'V', 'A', 'LA', 'PA'])  # Exclude categorical variables

# Add a constant for intercept
X = df_numeric.copy()
X['Intercept'] = 1  # Required for VIF calculation

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Drop the intercept row for interpretation
vif_data = vif_data[vif_data["Feature"] != "Intercept"]

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

# Display results
vif_data[vif_data['VIF'] >= 10]

,Feature,VIF
0,HRk,16.490040
2,HW,inf
3,HL,inf
4,HOL,inf
5,HPTS%,68.188705
6,HSOW,13.459593
7,HSOL,14.435630
8,HSRS,4013.965945
9,HSOS,28.413536
10,HGF/G,2010.902521


In [11]:
# VIF FEATURE SELECTION

# Suppress RuntimeWarnings in this block
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Function to compute VIF and iteratively remove high VIF features
def calculate_vif(df, threshold=10):
    X = df.copy()
    X['Intercept'] = 1  # Required for VIF calculation
    
    while True:
        # Compute VIF for each feature
        vif_data = pd.DataFrame()
        vif_data["Feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        
        # Drop intercept row
        vif_data = vif_data[vif_data["Feature"] != "Intercept"]
        
        # Find the feature with the highest VIF
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break  # Stop if all VIF values are below the threshold
        
        # Identify the feature to remove
        feature_to_remove = vif_data.loc[vif_data["VIF"].idxmax(), "Feature"]
        print(f"Removing {feature_to_remove} with VIF {max_vif:.2f}")
        
        # Drop the feature with the highest VIF
        X = X.drop(columns=[feature_to_remove])
    
    return X.drop(columns=['Intercept'])  # Return dataframe without high-VIF features

# Run the VIF reduction process (CAP --> drop LCAP)
df_numeric_cap = df_cap.drop(columns=['H', 'V', 'A', 'PA'])
df_numeric_lcap = df_lcap.drop(columns=['H', 'V', 'LA'])

df_reduced_cap = calculate_vif(df_numeric_cap)

# Run the VIF reduction process (LCAP --> drop CAP)
df_reduced_lcap = calculate_vif(df_numeric_lcap)

print(df_reduced_cap.columns)
print(df_reduced_lcap.columns)

selected_feats_cap_vif = df_reduced_cap.columns
selected_feats_lcap_vif = df_reduced_lcap.columns

# Re-enable warnings after this block
warnings.filterwarnings("default", category=RuntimeWarning)

Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.78
Removing HSRS with VIF 4007.60
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 62.99
Removing VW with VIF 32.97
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.06
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Removing HW with VIF inf
Removing MDAY with VIF inf
Removing VSRS with VIF 4049.76
Removing HSRS with VIF 4007.73
Removing VPPA with VIF 79.58
Removing VPTS% with VIF 63.58
Removing HPTS% with VIF 63.01
Removing VW with VIF 32.96
Removing VL with VIF 22.49
Removing VPPO with VIF 19.32
Removing HPPO with VIF 18.05
Removing HL with VIF 17.27
Removing VGA/G with VIF 10.36
Removing HGA/G with VIF 10.28
Index(['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HPP', 'HPPA',
       'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO',
       'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSO

In [12]:
# LASSO stability selection
def stability_selection_lasso(dep, X, y, alpha_range=np.logspace(-4, 1, 10), 
                              n_resampling=100, selection_threshold=0.5, name_feats=''):
    """
    Performs LASSO-based stability selection, finds the best alpha, and reports RMSE/MAE/R² for train/test (averaged over all resampling iterations).
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        alpha_range (array-like): List of alpha values to test.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be selected in.

    Returns:
        best_alpha (float): Optimal alpha with lowest RMSE.
        selected_features (list): Names of selected features using best_alpha.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """
    # Split into train/test sets (y is not scaled)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Standardize X only
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    results = []
    warning_alphas = []
    model_stats = {}

    for alpha in alpha_range:
        selection_counts = np.zeros(X.shape[1])
        
        # Initialize statistics to accumulate over iterations
        rmse_train_all = []
        rmse_test_all = []
        mae_train_all = []
        mae_test_all = []
        r2_train_all = []
        r2_test_all = []

        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            
            for _ in range(n_resampling):
                # bootstrap resampling
                X_sample, y_sample = resample(X_train_scaled, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)
                model = Lasso(alpha=alpha)
                model.fit(X_sample, y_sample)
                selection_counts += (model.coef_ != 0)

                # Make predictions
                y_train_pred = model.predict(X_train_scaled)
                y_test_pred = model.predict(X_test_scaled)

                # Compute statistics for this resample
                stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
                
                # Collect stats for averaging
                rmse_train_all.append(stats["RMSE Train"])
                rmse_test_all.append(stats["RMSE Test"])
                mae_train_all.append(stats["MAE Train"])
                mae_test_all.append(stats["MAE Test"])
                r2_train_all.append(stats["R² Train"])
                r2_test_all.append(stats["R² Test"])

            # Average the statistics over all resampling iterations
            avg_rmse_train = np.mean(rmse_train_all)
            avg_rmse_test = np.mean(rmse_test_all)
            avg_mae_train = np.mean(mae_train_all)
            avg_mae_test = np.mean(mae_test_all)
            avg_r2_train = np.mean(r2_train_all)
            avg_r2_test = np.mean(r2_test_all)

            # Store the averaged stats for this alpha
            model_stats[alpha] = {
                "RMSE Train": avg_rmse_train,
                "RMSE Test": avg_rmse_test,
                "MAE Train": avg_mae_train,
                "MAE Test": avg_mae_test,
                "R² Train": avg_r2_train,
                "R² Test": avg_r2_test
            }

        # Compute feature selection stability
        selection_frequencies = selection_counts / n_resampling
        selected_features = np.where(selection_frequencies >= selection_threshold)[0]
        num_selected = len(selected_features)

        results.append((alpha, num_selected, selected_features))

    # Extract alphas and feature counts
    alphas, feature_counts, feature_indices_list = zip(*results)

    # Plot feature count vs. alpha
    plt.figure(figsize=(12, 6))  # Increased figure size for better spacing
    plt.subplot(1, 2, 1)
    plt.plot(alphas, feature_counts, marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('Number of Features Selected')
    plt.title('Number of Features Selected for Different Alpha Values')
    
    # Plot RMSE vs. Alpha (use averaged RMSE Test)
    plt.subplot(1, 2, 2)
    plt.plot(model_stats.keys(), [stat["RMSE Test"] for stat in model_stats.values()], marker='o', linestyle='-', color='red')
    plt.xscale('log')
    plt.xlabel('Alpha')
    plt.ylabel('RMSE (Test)')
    plt.title('Test RMSE for Different Alpha Values')
    
    # Adjust layout for spacing between subplots
    plt.subplots_adjust(wspace=0.3, hspace=0.2)  # Increase horizontal and vertical space
    
    # Save the plot with a dynamic filename
    plt.savefig(f"LASSO_stability_{dep}_{name_feats}.png", dpi=300, bbox_inches='tight')
    
    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    best_alpha = min(model_stats, key=lambda a: model_stats[a]["RMSE Test"])

    # Recompute selection counts for best alpha
    selection_counts_best = np.zeros(X.shape[1])
    for _ in range(n_resampling):
        X_sample, y_sample = resample(X_train_scaled, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)
        model = Lasso(alpha=best_alpha)
        model.fit(X_sample, y_sample)
        selection_counts_best += (model.coef_ != 0)

    selection_frequencies_best = selection_counts_best / n_resampling
    selected_feature_indices = np.where(selection_frequencies_best >= selection_threshold)[0]
    selected_feature_names = X.columns[selected_feature_indices]

    # --- Plot bar chart ---
    plt.figure(figsize=(12, 6))
    plt.bar(X.columns, selection_frequencies_best, color="red")
    plt.axhline(selection_threshold, color="black", linestyle="--", label="Selection Threshold")
    plt.xticks(rotation=90)
    plt.xlabel("Feature")
    plt.ylabel("Selection Frequency")
    plt.title(f"LASSO Feature Stability for {dep} (α = {best_alpha:.4g})")
    plt.legend()
    plt.savefig(f"LASSO_stability_barchart_{dep}_{name_feats}.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # Convert indices to feature names
    selected_feature_names = X.columns[selected_feature_indices]

    print(f"Optimal alpha: {best_alpha}")
    print(f"Number of features selected: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    # Print statistics for best alpha
    best_stats = model_stats[best_alpha]
    print("\nPerformance Metrics (Train/Test):")
    for metric, value in best_stats.items():
        print(f"{metric}: {value:.4f}")

    # Print alphas that triggered convergence warnings
    if warning_alphas:
        print(f"\n⚠️ Convergence warnings occurred for alpha values: {warning_alphas}")

    return best_alpha, selected_feature_names, best_stats

In [13]:
# all features
# tends to not converge because of high degree of multicollinearity

print('Dependent variable: A')
alpha_cap_a, selected_feat_cap_a_lasso, best_stats_a = stability_selection_lasso('A', X_cap_a, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 1, 10), 
                                                                                 name_feats = 'full_list')

Dependent variable: A
Optimal alpha: 3.593813663804626
Number of features selected: 50
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.6284
RMSE Test: 1311.4428
MAE Train: 918.3517
MAE Test: 917.6337
R² Train: 0.5477
R² Test: 0.5249


In [14]:
# all featuress
print('Dependent variable: PA')
alpha_cap_pa, selected_feat_cap_pa_lasso, best_stats_pa = stability_selection_lasso('PA', X_cap_pa, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -3, 10),
                                                                                    name_feats = 'full_list')

Dependent variable: PA
Optimal alpha: 0.0001291549665014884
Number of features selected: 52
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VL', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VGA/G', 'VPP', 'VPPO', 'VPPA', 'VPPOA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1287.5812
RMSE Test: 1311.5812
MAE Train: 922.0697
MAE Test: 921.2737
R² Train: 0.5477
R² Test: 0.5248


In [15]:
# all features
print('Dependent variable: LA')
alpha_lcap, selected_feat_lcap_lasso, best_stats_la = stability_selection_lasso('LA', X_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-4, -3, 10),
                                                                                name_feats = 'full_list')

Dependent variable: LA
Optimal alpha: 0.0002782559402207126
Number of features selected: 47
Selected Features: ['HRk', 'HAvAge', 'HW', 'HL', 'HOL', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGA/G', 'HPP', 'HPPO', 'HPPA', 'HPPOA', 'HSH', 'HSHA', 'HPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSOW', 'VSOL', 'VSOS', 'VPP', 'VPPA', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'MDAY', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'LCAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1291.9437
RMSE Test: 1314.4763
MAE Train: 941.4653
MAE Test: 940.6836
R² Train: 0.5446
R² Test: 0.5227


In [16]:
# vif features
print('Dependent variable: A')
alpha_cap_a_vif, selected_feat_cap_a_lasso_vif, best_stats_a_vif = stability_selection_lasso('A', df_reduced_cap, y_cap_a, 
                                                                                 alpha_range = np.logspace(-1, 2, 10),
                                                                                 name_feats = 'vif')

print('\nDependent variable: PA')
alpha_cap_pa_vif, selected_feat_cap_pa_lasso_vif, best_stats_pa_vif = stability_selection_lasso('PA', df_reduced_cap, y_cap_pa, 
                                                                                    alpha_range = np.logspace(-5, -2, 10),
                                                                                    name_feats = 'vif')

print('\nDependent variable: LA')
alpha_lcap_vif, selected_feat_lcap_lasso_vif, best_stats_la_vif = stability_selection_lasso('LA', df_reduced_lcap, y_lcap, 
                                                                                alpha_range = np.logspace(-5, -2, 10),
                                                                                name_feats = 'vif')

Dependent variable: A
Optimal alpha: 10.0
Number of features selected: 36
Selected Features: ['HRk', 'HAvAge', 'HOL', 'HSOW', 'HSOL', 'HSOS', 'HGF/G', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge', 'VOL', 'VSOW', 'VSOS', 'VGF/G', 'VPPOA', 'VSH', 'VSHA', 'VoPIM/G', 'VS%', 'VSV%', 'VCAN', 'TDAY', 'WDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1310.5400
RMSE Test: 1329.9340
MAE Train: 936.6449
MAE Test: 937.3076
R² Train: 0.5314
R² Test: 0.5114

Dependent variable: PA
Optimal alpha: 0.001
Number of features selected: 25
Selected Features: ['HRk', 'HAvAge', 'HSOW', 'HSOS', 'HSH', 'HSHA', 'HoPIM/G', 'HS%', 'HSO', 'HCAN', 'VRk', 'VPPOA', 'VSH', 'VSHA', 'VoPIM/G', 'VSV%', 'VCAN', 'TDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'HSTAR', 'VSTAR']

Performance Metrics (Train/Test):
RMSE Train: 1313.5740
RMSE Test: 1329.8223
MAE Train: 939.8500
MAE Test: 939.1487
R² Train: 0.5292
R² Test: 0.5115

Dependent v

In [17]:
def stability_selection_rf(dep, X, y, n_estimators=100, n_resampling=100, selection_threshold=0.5, importance_threshold=0.5):
    """
    Performs stability selection for feature importance using Random Forest.
    
    Parameters:
        dep (str): Dependent variable type ('A', 'LA', or 'PA').
        X (DataFrame): Feature matrix.
        y (Series): Target variable.
        n_estimators (int): Number of trees in Random Forest.
        n_resampling (int): Number of resampling iterations.
        selection_threshold (float): Minimum fraction of resamples a feature must be important in.
        importance_threshold (float): Fraction of most important features to consider in each iteration.

    Returns:
        selected_features (list): Names of selected features.
        stats (dict): Average RMSE, MAE, R² for train/test.
    """

    # Split into train/test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Track feature importance frequency
    feature_importance_counts = np.zeros(X.shape[1])

    # Initialize statistics tracking
    rmse_train_all, rmse_test_all = [], []
    mae_train_all, mae_test_all = [], []
    r2_train_all, r2_test_all = [], []

    for _ in range(n_resampling):
        # Bootstrap resample
        X_sample, y_sample = resample(X_train, y_train, n_samples=int(0.7 * len(y_train)), random_state=None)

        # Train Random Forest
        rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
        rf.fit(X_sample, y_sample)

        # Determine number of features to consider based on importance_threshold
        num_top_features = max(1, int(importance_threshold * len(X.columns)))  # Ensure at least 1 feature is selected
        importance_ranking = np.argsort(rf.feature_importances_)[::-1]  # Indices of features sorted by importance
        top_features = importance_ranking[:num_top_features]  
        feature_importance_counts[top_features] += 1

        # Make predictions
        y_train_pred = rf.predict(X_train)
        y_test_pred = rf.predict(X_test)

        # Compute statistics for this resample
        stats = compute_stats(dep, X_train, X_test, y_train, y_train_pred, y_test, y_test_pred)
        
        # Collect stats for averaging
        rmse_train_all.append(stats["RMSE Train"])
        rmse_test_all.append(stats["RMSE Test"])
        mae_train_all.append(stats["MAE Train"])
        mae_test_all.append(stats["MAE Test"])
        r2_train_all.append(stats["R² Train"])
        r2_test_all.append(stats["R² Test"])

    # Compute stability selection scores
    selection_frequencies = feature_importance_counts / n_resampling
    selected_features_indices = np.where(selection_frequencies >= selection_threshold)[0]
    selected_feature_names = X.columns[selected_features_indices]

    # Compute averaged statistics
    stats = {
        "RMSE Train": np.mean(rmse_train_all),
        "RMSE Test": np.mean(rmse_test_all),
        "MAE Train": np.mean(mae_train_all),
        "MAE Test": np.mean(mae_test_all),
        "R² Train": np.mean(r2_train_all),
        "R² Test": np.mean(r2_test_all),
    }

    # Plot feature stability selection (VERTICAL bar chart)
    plt.figure(figsize=(12, 6))
    plt.bar(X.columns, selection_frequencies, color="red")
    plt.axhline(selection_threshold, color="black", linestyle="--", label="Selection Threshold")
    
    plt.xticks(rotation=90)  # Rotate x-axis labels for readability
    plt.xlabel("Feature")
    plt.ylabel("Selection Frequency")
    plt.title(f"Feature Selection Stability for {dep}")
    plt.legend()

    plt.savefig(f"RF_stability_{dep}.png", dpi=300, bbox_inches='tight')

    # Close the plot to free up memory and avoid duplicate plots
    plt.close()

    print(f"Number of selected features: {len(selected_feature_names)}")
    print("Selected Features:", selected_feature_names.tolist())

    print("\nPerformance Metrics (Train/Test):")
    for metric, value in stats.items():
        print(f"{metric}: {value:.4f}")

    return selected_feature_names, stats

In [ ]:
print('Dependent variable: A')
selected_feat_cap_a_rf, best_stats_a_rf = stability_selection_rf('A', X_cap_a, y_cap_a)

Dependent variable: A


In [ ]:
print('Dependent variable: PA')
selected_feat_cap_pa_rf, best_stats_pa_rf = stability_selection_rf('PA', X_cap_pa, y_cap_pa)

In [ ]:
print('Dependent variable: LA')
selected_feat_lcap_rf, best_stats_la_rf = stability_selection_rf('LA', X_lcap, y_lcap)

In [ ]:
#### OLS WITH LASSO SELECTED FEATS

# A

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

In [ ]:
# PA

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

In [ ]:
# LA

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_lasso]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

In [ ]:
#### OLS WITH RF SELECTED FEATS

# A

# Subset the data to include only the selected features
X_selected = X_cap_a[selected_feat_cap_a_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_a, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

In [ ]:
#### OLS WITH RF SELECTED FEATS

# PA

# Subset the data to include only the selected features
X_selected = X_cap_pa[selected_feat_cap_pa_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_cap_pa, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())

In [ ]:
#### OLS WITH RF SELECTED FEATS

# LA

# Subset the data to include only the selected features
X_selected = X_lcap[selected_feat_lcap_rf]

# Add a constant term for the intercept
X_selected = sm.add_constant(X_selected)

# Run the OLS regression
ols_model = sm.OLS(y_lcap, X_selected).fit()

# Print the summary of the regression
display(ols_model.summary())